# YOLOv26s/m Segmentation Training Notebook  
## Speed Stack Cup 3-Class Dataset: `fallen-cup`, `upright-cup`, `mouth-up-cup`

이 노트북은 Google Drive에 업로드된 Roboflow COCO Segmentation zip 파일을 사용해서 **YOLOv26s-seg**와 **YOLOv26m-seg**를 순차적으로 학습합니다.

### 핵심 설정
- 학습 모델: `YOLOv26s-seg`, `YOLOv26m-seg`
- 클래스: `fallen-cup`, `upright-cup`, `mouth-up-cup`
- hard-negative/background 이미지 지원
- 빨간 컵이 dataset에 이미 포함되어 있으므로 red recolor augmentation은 일부 이미지만 약하게 적용
- 결과는 Google Drive에 자동 저장

## 1. Environment setup

In [ ]:
!nvidia-smi

!pip install -q -U ultralytics opencv-python-headless pycocotools pyyaml matplotlib openpyxl albumentations
!pip install -q "pandas==2.2.2"

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, gc, json, math, yaml, shutil, zipfile, random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
from ultralytics import YOLO
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

Mon Jun  8 14:50:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Global configuration

In [ ]:
# =========================
# User config
# =========================

# Google Drive에 업로드한 Roboflow COCO Segmentation zip 파일명
DATASET_ZIP_NAME = "hand-eye-view-speed-stack-cup.v2-mouth-up-cup.coco-segmentation.zip"

# zip 파일이 MyDrive 바로 아래에 있으면 이 경로를 그대로 사용합니다.
# 다른 폴더에 있다면 이 경로만 수정하세요.
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive") / DATASET_ZIP_NAME

# DRIVE_ZIP_PATH가 없으면 MyDrive 전체에서 같은 파일명을 자동 검색
AUTO_FIND_ZIP_IN_DRIVE = True

IMG_SIZE = 1280
EPOCHS = 250
PATIENCE = 60

# small, medium만 학습
MODEL_SIZES = ["s", "m"]
MODEL_NAME_MAP = {
    "n": "YOLOv26n-seg",
    "s": "YOLOv26s-seg",
    "m": "YOLOv26m-seg",
    "l": "YOLOv26l-seg",
}

EXPECTED_CLASSES = ["fallen-cup", "upright-cup", "mouth-up-cup"]

# =========================
# Offline augmentation config
# =========================

USE_OFFLINE_AUGMENTATION = True

# 원본 train 이미지 1장당 geometric/photometric augmentation 복제본 수
# 고정 작업영역이므로 1 권장
AUG_COPIES_PER_IMAGE = 1

# hard-negative 이미지도 geometric/photometric augmentation할지 여부
AUGMENT_NEGATIVE_IMAGES = True

# 빨간 컵이 실제 dataset에 있으므로 red recolor는 약하게만 적용
USE_RED_RECOLOR_AUG = True

# 라벨이 있는 train 이미지 중 약 25%만 red recolor 복제본 생성
RED_RECOLOR_PROB = 0.25
RED_COPIES_PER_SELECTED_IMAGE = 1

RED_HUE_OPENCV = 0
RED_HUE_JITTER = 6
RED_APPLY_PHOTO_AUG = True

REBUILD_AUGMENTED_DATASET = True
SAVE_AUGMENTED_DATASET_TO_DRIVE = False

AUG_SEED = 42
random.seed(AUG_SEED)
np.random.seed(AUG_SEED)

# =========================
# YOLO training config
# =========================

# A100 기준 자동 batch. OOM이 나면 8, 6, 4 같은 정수로 낮추세요.
TRAIN_BATCH = 0.60
EVAL_BATCH = 8

# cache='ram'은 메모리를 많이 사용하므로 기본 False
CACHE_MODE = False

PRED_CONF = 0.25
PRED_IOU = 0.70

SKIP_TRAIN_IF_DRIVE_BEST_EXISTS = False
SAVE_RUN_ZIP = True

# =========================
# Local paths
# =========================

LOCAL_ZIP_PATH = Path("/content") / DATASET_ZIP_NAME
LOCAL_EXTRACT_ROOT = Path("/content/roboflow_coco_zip_extracted")
LOCAL_COCO_DIR = Path("/content/roboflow_coco_dataset")

YOLO_DATASET_DIR = Path(f"/content/speedstack_3class_yolo_seg_{IMG_SIZE}")
DATA_YAML = YOLO_DATASET_DIR / "data.yaml"

AUG_TAG = f"geom{AUG_COPIES_PER_IMAGE}_redp{int(RED_RECOLOR_PROB * 100)}"
AUGMENTED_YOLO_DATASET_DIR = Path(f"/content/speedstack_3class_yolo_seg_{IMG_SIZE}_{AUG_TAG}")
AUGMENTED_DATA_YAML = AUGMENTED_YOLO_DATASET_DIR / "data.yaml"

RUN_PROJECT = Path("/content/runs/segment")
EVAL_PROJECT = Path("/content/runs/segment_eval")
PRED_PROJECT = Path("/content/runs/segment_predict")

EXPERIMENT_TAG = f"3class_lightaug_{AUG_TAG}_sm"
DRIVE_SAVE_DIR = Path(f"/content/drive/MyDrive/yolo26sm_3class_speedstack_result_epoch{EPOCHS}_{EXPERIMENT_TAG}")

DRIVE_WEIGHTS_DIR = DRIVE_SAVE_DIR / "weights"
DRIVE_RUNS_DIR = DRIVE_SAVE_DIR / "runs_zip"
DRIVE_PRED_DIR = DRIVE_SAVE_DIR / "predict_images"
DRIVE_TABLE_DIR = DRIVE_SAVE_DIR / "tables"
DRIVE_AUG_DATASET_DIR = DRIVE_SAVE_DIR / "augmented_dataset"

for d in [DRIVE_SAVE_DIR, DRIVE_WEIGHTS_DIR, DRIVE_RUNS_DIR, DRIVE_PRED_DIR, DRIVE_TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("DATASET_ZIP_NAME:", DATASET_ZIP_NAME)
print("DRIVE_ZIP_PATH:", DRIVE_ZIP_PATH)
print("MODEL_SIZES:", MODEL_SIZES)
print("IMG_SIZE:", IMG_SIZE)
print("EPOCHS:", EPOCHS)
print("USE_OFFLINE_AUGMENTATION:", USE_OFFLINE_AUGMENTATION)
print("AUG_COPIES_PER_IMAGE:", AUG_COPIES_PER_IMAGE)
print("USE_RED_RECOLOR_AUG:", USE_RED_RECOLOR_AUG)
print("RED_RECOLOR_PROB:", RED_RECOLOR_PROB)
print("EXPERIMENT_TAG:", EXPERIMENT_TAG)
print("DRIVE_SAVE_DIR:", DRIVE_SAVE_DIR)
print("DEVICE:", DEVICE)

DATASET_ZIP_NAME: hand-eye-view-speed-stack-cup.v2-mouth-up-cup.coco-segmentation.zip
DRIVE_ZIP_PATH: /content/drive/MyDrive/hand-eye-view-speed-stack-cup.v2-mouth-up-cup.coco-segmentation.zip
MODEL_SIZES: ['s', 'm']
IMG_SIZE: 1280
EPOCHS: 250
USE_OFFLINE_AUGMENTATION: True
AUG_COPIES_PER_IMAGE: 1
USE_RED_RECOLOR_AUG: True
RED_RECOLOR_PROB: 0.25
EXPERIMENT_TAG: 3class_lightaug_geom1_redp25_sm
DRIVE_SAVE_DIR: /content/drive/MyDrive/yolo26sm_3class_speedstack_result_epoch250_3class_lightaug_geom1_redp25_sm
DEVICE: 0


## 3. Copy and extract Roboflow COCO zip from Google Drive

In [ ]:
# =========================
# Locate zip file
# =========================

if not DRIVE_ZIP_PATH.exists() and AUTO_FIND_ZIP_IN_DRIVE:
    print("[INFO] DRIVE_ZIP_PATH not found. Searching MyDrive recursively...")
    matches = list(Path("/content/drive/MyDrive").rglob(DATASET_ZIP_NAME))
    if len(matches) > 0:
        DRIVE_ZIP_PATH = matches[0]
        print("[FOUND]", DRIVE_ZIP_PATH)
    else:
        print("[NOT FOUND] No matching zip file found in MyDrive.")

assert DRIVE_ZIP_PATH.exists(), f"Dataset zip not found: {DRIVE_ZIP_PATH}"

# =========================
# Copy zip to local
# =========================

if LOCAL_ZIP_PATH.exists():
    LOCAL_ZIP_PATH.unlink()

shutil.copy2(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
print("Copied zip to:", LOCAL_ZIP_PATH)

# =========================
# Extract zip
# =========================

if LOCAL_EXTRACT_ROOT.exists():
    shutil.rmtree(LOCAL_EXTRACT_ROOT)
LOCAL_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as zf:
    zf.extractall(LOCAL_EXTRACT_ROOT)

print("Extracted to:", LOCAL_EXTRACT_ROOT)

print("\nTop-level extracted files/folders:")
for p in sorted(LOCAL_EXTRACT_ROOT.iterdir()):
    print(" -", p.name)

Copied zip to: /content/hand-eye-view-speed-stack-cup.v2-mouth-up-cup.coco-segmentation.zip
Extracted to: /content/roboflow_coco_zip_extracted

Top-level extracted files/folders:
 - README.dataset.txt
 - README.roboflow.txt
 - test
 - train
 - valid


In [ ]:
# =========================
# Detect actual COCO dataset root
# =========================

def has_coco_splits(root: Path):
    for split in ["train", "valid", "val", "test"]:
        if (root / split / "_annotations.coco.json").exists():
            return True
    return False

candidate_roots = [LOCAL_EXTRACT_ROOT]
candidate_roots += [p for p in LOCAL_EXTRACT_ROOT.iterdir() if p.is_dir()]

detected = None
for c in candidate_roots:
    if has_coco_splits(c):
        detected = c
        break

assert detected is not None, "Could not find COCO split folders containing _annotations.coco.json"

if LOCAL_COCO_DIR.exists():
    shutil.rmtree(LOCAL_COCO_DIR)

shutil.copytree(detected, LOCAL_COCO_DIR)

print("Detected COCO dataset root:", detected)
print("Copied normalized COCO dataset to:", LOCAL_COCO_DIR)

print("\nCOCO split folders:")
for split in ["train", "valid", "val", "test"]:
    ann = LOCAL_COCO_DIR / split / "_annotations.coco.json"
    print(split, "exists:", ann.exists(), ann)

Detected COCO dataset root: /content/roboflow_coco_zip_extracted
Copied normalized COCO dataset to: /content/roboflow_coco_dataset

COCO split folders:
train exists: True /content/roboflow_coco_dataset/train/_annotations.coco.json
valid exists: True /content/roboflow_coco_dataset/valid/_annotations.coco.json
val exists: False /content/roboflow_coco_dataset/val/_annotations.coco.json
test exists: True /content/roboflow_coco_dataset/test/_annotations.coco.json


## 4. Check COCO dataset structure and classes

In [ ]:
POSSIBLE_SPLITS = ["train", "valid", "val", "test"]

split_jsons = {}
for split in POSSIBLE_SPLITS:
    ann_path = LOCAL_COCO_DIR / split / "_annotations.coco.json"
    if ann_path.exists():
        split_jsons[split] = ann_path

print("Found splits:")
for split, path in split_jsons.items():
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n[{split}]")
    print("json:", path)
    print("images:", len(data.get("images", [])))
    print("annotations:", len(data.get("annotations", [])))
    print("categories:", [c.get("name") for c in data.get("categories", [])])

assert "train" in split_jsons, "train split is required."
assert ("valid" in split_jsons) or ("val" in split_jsons), "valid/val split is required."
assert "test" in split_jsons, "test split is recommended and required by this notebook."

with open(split_jsons["train"], "r", encoding="utf-8") as f:
    train_coco_raw = json.load(f)

category_names = [c["name"] for c in sorted(train_coco_raw["categories"], key=lambda x: x["id"])]
print("\nDetected category names:", category_names)
print("Expected class names:", EXPECTED_CLASSES)

missing = set(EXPECTED_CLASSES) - set(category_names)
extra = set(category_names) - set(EXPECTED_CLASSES)

if missing:
    print("[WARNING] Missing expected classes:", missing)
if extra:
    print("[WARNING] Extra classes in dataset:", extra)

assert len(category_names) >= 3, "Expected at least 3 classes."

Found splits:

[train]
json: /content/roboflow_coco_dataset/train/_annotations.coco.json
images: 1047
annotations: 3417
categories: ['hand-eye-view-speed-stack-cup', 'fallen-cup', 'mouth-up-cup', 'upright-cup']

[valid]
json: /content/roboflow_coco_dataset/valid/_annotations.coco.json
images: 100
annotations: 306
categories: ['hand-eye-view-speed-stack-cup', 'fallen-cup', 'mouth-up-cup', 'upright-cup']

[test]
json: /content/roboflow_coco_dataset/test/_annotations.coco.json
images: 50
annotations: 152
categories: ['hand-eye-view-speed-stack-cup', 'fallen-cup', 'mouth-up-cup', 'upright-cup']

Detected category names: ['hand-eye-view-speed-stack-cup', 'fallen-cup', 'mouth-up-cup', 'upright-cup']
Expected class names: ['fallen-cup', 'upright-cup', 'mouth-up-cup']
[WARNING] Extra classes in dataset: {'hand-eye-view-speed-stack-cup'}


## 5. Convert COCO segmentation to YOLO segmentation

In [ ]:
# =========================
# COCO segmentation -> YOLO segmentation converter
# =========================

def find_image_path(split_dir: Path, file_name: str):
    candidates = [
        split_dir / file_name,
        split_dir / Path(file_name).name,
    ]

    for p in candidates:
        if p.exists():
            return p

    matches = list(split_dir.rglob(Path(file_name).name))
    if matches:
        return matches[0]

    return None


def polygon_to_yolo_coords(poly, width, height):
    arr = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
    if len(arr) < 3:
        return None

    arr[:, 0] = np.clip(arr[:, 0] / width, 0.0, 1.0)
    arr[:, 1] = np.clip(arr[:, 1] / height, 0.0, 1.0)

    coords = arr.reshape(-1)
    if len(coords) < 6:
        return None

    return coords


def rle_to_polygons(segmentation, width, height, min_area=20, epsilon_ratio=0.002):
    if isinstance(segmentation, list):
        return segmentation

    try:
        if isinstance(segmentation.get("counts"), list):
            rle = maskUtils.frPyObjects(segmentation, height, width)
        else:
            rle = segmentation

        mask = maskUtils.decode(rle)
        if mask.ndim == 3:
            mask = np.any(mask, axis=2).astype(np.uint8)

        mask_u8 = (mask > 0).astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        polygons = []
        for contour in contours:
            if cv2.contourArea(contour) < min_area:
                continue
            epsilon = epsilon_ratio * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True)
            pts = approx.reshape(-1, 2)
            if len(pts) >= 3:
                polygons.append(pts.reshape(-1).astype(float).tolist())
        return polygons

    except Exception as e:
        print("[WARN] failed to decode RLE:", e)
        return []


def convert_split(coco_json_path: Path, split_out_name: str):
    coco = COCO(str(coco_json_path))

    split_dir = coco_json_path.parent
    img_out_dir = YOLO_DATASET_DIR / "images" / split_out_name
    lbl_out_dir = YOLO_DATASET_DIR / "labels" / split_out_name

    img_out_dir.mkdir(parents=True, exist_ok=True)
    lbl_out_dir.mkdir(parents=True, exist_ok=True)

    cats = coco.loadCats(coco.getCatIds())
    cats_sorted = sorted(cats, key=lambda c: c["id"])
    cat_id_to_yolo = {cat["id"]: i for i, cat in enumerate(cats_sorted)}
    names = [cat["name"] for cat in cats_sorted]

    img_ids = coco.getImgIds()

    copied_count = 0
    label_object_count = 0
    null_image_count = 0
    missing_image_count = 0

    for img_id in img_ids:
        img_info = coco.loadImgs([img_id])[0]
        file_name = img_info["file_name"]
        width = int(img_info["width"])
        height = int(img_info["height"])

        src_img = find_image_path(split_dir, file_name)
        if src_img is None:
            print("[WARN] image not found:", file_name)
            missing_image_count += 1
            continue

        dst_img = img_out_dir / Path(file_name).name
        shutil.copy2(src_img, dst_img)
        copied_count += 1

        ann_ids = coco.getAnnIds(imgIds=[img_id])
        anns = coco.loadAnns(ann_ids)

        lines = []

        for ann in anns:
            if ann.get("iscrowd", 0) == 1:
                continue

            cat_id = ann["category_id"]
            if cat_id not in cat_id_to_yolo:
                continue

            cls = cat_id_to_yolo[cat_id]
            segmentation = ann.get("segmentation", [])

            polygons = rle_to_polygons(segmentation, width, height)

            for poly in polygons:
                if len(poly) < 6:
                    continue

                coords = polygon_to_yolo_coords(poly, width, height)
                if coords is None:
                    continue

                coord_str = " ".join([f"{x:.6f}" for x in coords])
                lines.append(f"{cls} {coord_str}")

        label_path = lbl_out_dir / f"{Path(file_name).stem}.txt"
        label_path.write_text("\n".join(lines), encoding="utf-8")

        if len(lines) == 0:
            null_image_count += 1
        else:
            label_object_count += len(lines)

    return {
        "split": split_out_name,
        "images": copied_count,
        "objects": label_object_count,
        "null_images": null_image_count,
        "missing_images": missing_image_count,
        "names": names,
    }


if YOLO_DATASET_DIR.exists():
    shutil.rmtree(YOLO_DATASET_DIR)

split_map = {}
if "train" in split_jsons:
    split_map["train"] = "train"
if "valid" in split_jsons:
    split_map["valid"] = "val"
elif "val" in split_jsons:
    split_map["val"] = "val"
if "test" in split_jsons:
    split_map["test"] = "test"

convert_summaries = []
names_ref = None

for coco_split, yolo_split in split_map.items():
    summary = convert_split(split_jsons[coco_split], yolo_split)
    convert_summaries.append(summary)
    names_ref = summary["names"]

print("========== CONVERSION SUMMARY ==========")
for s in convert_summaries:
    print(s)

DATA_YAML.parent.mkdir(parents=True, exist_ok=True)

data_yaml_dict = {
    "path": str(YOLO_DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(names_ref),
    "names": names_ref,
}

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml_dict, f, sort_keys=False, allow_unicode=True)

print("\nSaved data.yaml:")
print(DATA_YAML.read_text(encoding="utf-8"))

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
========== CONVERSION SUMMARY ==========
{'split': 'train', 'images': 1047, 'objects': 3562, 'null_images': 165, 'missing_images': 0, 'names': ['hand-eye-view-speed-stack-cup', 'fallen-cup', 'mouth-up-cup', 'upright-cup']}
{'split': 'val', 'images': 100, 'objects': 326, 'null_images': 14, 'missing_images': 0, 'names': ['hand-eye-view-speed-stack-cup', 'fallen-cup', 'mouth-up-cup', 'upright-cup']}
{'split': 'test', 'images': 50, 'objects': 160, 'null_images': 5, 'missing_images': 0, 'names': ['hand-eye-view-speed-stack-cup', 'fallen-cup', 'mouth-up-cup', 'upright-cup']}

Saved data.yaml:
path: /content/speedstack_3class_yolo_seg_1280
train: images/train
val: images/val
test: images/test
nc: 4
names:
- hand-eye-view-speed-stack-cup
- fallen-cu

## 6. Verify YOLO labels and null images

In [ ]:
def image_files_in(img_dir: Path):
    return sorted([
        p for p in img_dir.glob("*")
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ])


def count_images(img_dir: Path):
    return len(image_files_in(img_dir))


def verify_yolo_seg_labels(yolo_dataset_dir: Path, split="train"):
    img_dir = yolo_dataset_dir / "images" / split
    lbl_dir = yolo_dataset_dir / "labels" / split

    img_files = image_files_in(img_dir)
    label_files = sorted(lbl_dir.glob("*.txt"))

    empty_labels = []
    non_empty_labels = []
    missing_labels = []

    for img_path in img_files:
        label_path = lbl_dir / f"{img_path.stem}.txt"
        if not label_path.exists():
            missing_labels.append(img_path)
        elif label_path.read_text(encoding="utf-8").strip() == "":
            empty_labels.append(label_path)
        else:
            non_empty_labels.append(label_path)

    print(f"\n[{split}]")
    print("image count:", len(img_files))
    print("label file count:", len(label_files))
    print("non-empty labels:", len(non_empty_labels))
    print("empty/null labels:", len(empty_labels))
    print("missing label files:", len(missing_labels))

    max_cls = -1
    bad_lines = 0
    for lp in label_files:
        for line in lp.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
                bad_lines += 1
                continue
            cls = int(float(parts[0]))
            max_cls = max(max_cls, cls)

    print("max class id:", max_cls)
    print("bad lines:", bad_lines)


for split in ["train", "val", "test"]:
    verify_yolo_seg_labels(YOLO_DATASET_DIR, split)


[train]
image count: 1047
label file count: 1047
non-empty labels: 882
empty/null labels: 165
missing label files: 0
max class id: 3
bad lines: 0

[val]
image count: 100
label file count: 100
non-empty labels: 86
empty/null labels: 14
missing label files: 0
max class id: 3
bad lines: 0

[test]
image count: 50
label file count: 50
non-empty labels: 45
empty/null labels: 5
missing label files: 0
max class id: 3
bad lines: 0


## 7. Optional offline augmentation

현재 설정은 약한 augmentation입니다.

- 원본 train 이미지당 geometric/photometric augmentation 1장
- 라벨이 있는 train 이미지 중 약 25%에만 red recolor augmentation 1장
- hard-negative/null 이미지도 geometric/photometric augmentation 가능
- validation/test는 원본 그대로 유지

In [ ]:
# =========================
# Offline augmentation utilities
# =========================

import albumentations as A

def read_yolo_seg_label(label_path: Path):
    objects = []
    if not label_path.exists():
        return objects

    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue

        cls = int(float(parts[0]))
        coords = np.array([float(x) for x in parts[1:]], dtype=np.float32)

        if len(coords) % 2 == 1:
            coords = coords[:-1]

        if len(coords) < 6:
            continue

        poly = coords.reshape(-1, 2)
        poly = np.clip(poly, 0.0, 1.0)
        objects.append((cls, poly))

    return objects


def write_yolo_seg_label(label_path: Path, objects):
    lines = []

    for cls, poly in objects:
        if poly is None or len(poly) < 3:
            continue

        poly = np.asarray(poly, dtype=np.float32)
        poly = np.clip(poly, 0.0, 1.0)

        coords = []
        for x, y in poly:
            coords.append(f"{x:.6f}")
            coords.append(f"{y:.6f}")

        if len(coords) >= 6:
            lines.append(f"{cls} " + " ".join(coords))

    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text("\n".join(lines), encoding="utf-8")
    return len(lines)


def yolo_poly_to_mask(poly_norm, width, height):
    pts = np.asarray(poly_norm, dtype=np.float32).copy()
    pts[:, 0] *= width
    pts[:, 1] *= height
    pts = np.round(pts).astype(np.int32)

    mask = np.zeros((height, width), dtype=np.uint8)
    if len(pts) >= 3:
        cv2.fillPoly(mask, [pts], 1)
    return mask


def mask_to_yolo_polygon(mask, width, height, min_area=20, epsilon_ratio=0.002):
    mask_u8 = (mask > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    contour = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(contour)

    if area < min_area:
        return None

    epsilon = epsilon_ratio * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    pts = approx.reshape(-1, 2).astype(np.float32)

    if len(pts) < 3:
        return None

    pts[:, 0] = np.clip(pts[:, 0] / width, 0.0, 1.0)
    pts[:, 1] = np.clip(pts[:, 1] / height, 0.0, 1.0)

    return pts


def make_light_augmenter():
    return A.Compose(
        [
            A.Affine(
                translate_percent={"x": (-0.03, 0.03), "y": (-0.03, 0.03)},
                scale=(0.92, 1.08),
                rotate=(-8, 8),
                shear=(-2, 2),
                border_mode=cv2.BORDER_CONSTANT,
                fill=(114, 114, 114),
                fill_mask=0,
                p=0.80,
            ),
            A.OneOf(
                [
                    A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=1.0),
                    A.RandomGamma(gamma_limit=(90, 110), p=1.0),
                    A.CLAHE(clip_limit=(1.0, 2.0), tile_grid_size=(8, 8), p=1.0),
                ],
                p=0.60,
            ),
            A.HueSaturationValue(
                hue_shift_limit=4,
                sat_shift_limit=12,
                val_shift_limit=10,
                p=0.35,
            ),
            A.OneOf(
                [
                    A.GaussianBlur(blur_limit=(3, 3), p=1.0),
                    A.MotionBlur(blur_limit=(3, 3), p=1.0),
                    A.GaussNoise(std_range=(0.01, 0.03), p=1.0),
                ],
                p=0.15,
            ),
        ],
    )


def make_photo_only_augmenter():
    return A.Compose(
        [
            A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.6),
            A.HueSaturationValue(hue_shift_limit=2, sat_shift_limit=8, val_shift_limit=8, p=0.4),
        ]
    )


def recolor_cup_hue(image_bgr, masks, target_hue=RED_HUE_OPENCV, hue_jitter=RED_HUE_JITTER):
    if not masks:
        return image_bgr.copy()

    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    union = np.zeros(image_bgr.shape[:2], dtype=bool)

    for m in masks:
        union |= (np.asarray(m) > 0)

    if not union.any():
        return image_bgr.copy()

    h = int(target_hue + np.random.randint(-hue_jitter, hue_jitter + 1)) % 180
    hsv[..., 0][union] = h
    hsv[..., 1][union] = np.clip(hsv[..., 1][union].astype(np.int16) + 15, 0, 255).astype(np.uint8)

    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)


def augment_one_image(img_path: Path, lbl_path: Path, out_img_path: Path, out_lbl_path: Path, augmenter):
    image = cv2.imread(str(img_path))
    if image is None:
        return 0

    h, w = image.shape[:2]
    objects = read_yolo_seg_label(lbl_path)

    if len(objects) == 0:
        if not AUGMENT_NEGATIVE_IMAGES:
            return 0

        transformed = augmenter(image=image, masks=[])
        aug_img = transformed["image"]

        cv2.imwrite(str(out_img_path), aug_img)
        out_lbl_path.write_text("", encoding="utf-8")
        return 0

    masks = []
    cls_ids = []

    for cls, poly in objects:
        mask = yolo_poly_to_mask(poly, w, h)
        masks.append(mask)
        cls_ids.append(cls)

    transformed = augmenter(image=image, masks=masks)
    aug_img = transformed["image"]
    aug_masks = transformed["masks"]

    new_objects = []
    h2, w2 = aug_img.shape[:2]

    for cls, mask in zip(cls_ids, aug_masks):
        poly = mask_to_yolo_polygon(mask, w2, h2)
        if poly is not None:
            new_objects.append((cls, poly))

    cv2.imwrite(str(out_img_path), aug_img)
    return write_yolo_seg_label(out_lbl_path, new_objects)


def create_red_recolor_image(img_path: Path, lbl_path: Path, out_img_path: Path, out_lbl_path: Path):
    image = cv2.imread(str(img_path))
    if image is None:
        return 0

    h, w = image.shape[:2]
    objects = read_yolo_seg_label(lbl_path)

    if len(objects) == 0:
        return 0

    masks = []
    for cls, poly in objects:
        masks.append(yolo_poly_to_mask(poly, w, h))

    red_image = recolor_cup_hue(image, masks)

    if RED_APPLY_PHOTO_AUG:
        photo_aug = make_photo_only_augmenter()
        red_image = photo_aug(image=red_image)["image"]

    cv2.imwrite(str(out_img_path), red_image)
    shutil.copy2(lbl_path, out_lbl_path)
    return len(objects)

In [ ]:
# =========================
# Create augmented YOLO dataset
# =========================

if not USE_OFFLINE_AUGMENTATION:
    DATA_YAML_FOR_TRAIN = DATA_YAML
    print("Using original YOLO dataset:", YOLO_DATASET_DIR)

else:
    if REBUILD_AUGMENTED_DATASET and AUGMENTED_YOLO_DATASET_DIR.exists():
        shutil.rmtree(AUGMENTED_YOLO_DATASET_DIR)

    if not AUGMENTED_YOLO_DATASET_DIR.exists():
        print("Creating augmented dataset:", AUGMENTED_YOLO_DATASET_DIR)

        shutil.copytree(YOLO_DATASET_DIR, AUGMENTED_YOLO_DATASET_DIR)

        train_img_dir = YOLO_DATASET_DIR / "images" / "train"
        train_lbl_dir = YOLO_DATASET_DIR / "labels" / "train"

        out_train_img_dir = AUGMENTED_YOLO_DATASET_DIR / "images" / "train"
        out_train_lbl_dir = AUGMENTED_YOLO_DATASET_DIR / "labels" / "train"

        augmenter = make_light_augmenter()
        original_train_images = image_files_in(train_img_dir)

        geom_count = 0
        geom_obj_count = 0
        red_count = 0
        red_obj_count = 0

        for img_path in original_train_images:
            lbl_path = train_lbl_dir / f"{img_path.stem}.txt"
            objects = read_yolo_seg_label(lbl_path)

            for k in range(AUG_COPIES_PER_IMAGE):
                out_img_path = out_train_img_dir / f"{img_path.stem}_aug{k+1:02d}{img_path.suffix}"
                out_lbl_path = out_train_lbl_dir / f"{img_path.stem}_aug{k+1:02d}.txt"

                obj_num = augment_one_image(img_path, lbl_path, out_img_path, out_lbl_path, augmenter)
                geom_count += 1
                geom_obj_count += obj_num

            if USE_RED_RECOLOR_AUG and len(objects) > 0 and random.random() < RED_RECOLOR_PROB:
                for k in range(RED_COPIES_PER_SELECTED_IMAGE):
                    out_img_path = out_train_img_dir / f"{img_path.stem}_redlite{k+1:02d}{img_path.suffix}"
                    out_lbl_path = out_train_lbl_dir / f"{img_path.stem}_redlite{k+1:02d}.txt"

                    obj_num = create_red_recolor_image(img_path, lbl_path, out_img_path, out_lbl_path)
                    red_count += 1
                    red_obj_count += obj_num

        with open(DATA_YAML, "r", encoding="utf-8") as f:
            base_yaml = yaml.safe_load(f)

        base_yaml["path"] = str(AUGMENTED_YOLO_DATASET_DIR)

        with open(AUGMENTED_DATA_YAML, "w", encoding="utf-8") as f:
            yaml.safe_dump(base_yaml, f, sort_keys=False, allow_unicode=True)

        print("\n========== AUGMENTATION SUMMARY ==========")
        print("original_train_images:", len(original_train_images))
        print("geometric_aug_images:", geom_count)
        print("red_recolor_images:", red_count)
        print("total_train_images:", count_images(out_train_img_dir))
        print("geometric_aug_objects:", geom_obj_count)
        print("red_recolor_objects:", red_obj_count)
        print("AUGMENTED_DATA_YAML:", AUGMENTED_DATA_YAML)
        print(AUGMENTED_DATA_YAML.read_text(encoding="utf-8"))

        if SAVE_AUGMENTED_DATASET_TO_DRIVE:
            if DRIVE_AUG_DATASET_DIR.exists():
                shutil.rmtree(DRIVE_AUG_DATASET_DIR)
            shutil.copytree(AUGMENTED_YOLO_DATASET_DIR, DRIVE_AUG_DATASET_DIR)
            print("Saved augmented dataset to Drive:", DRIVE_AUG_DATASET_DIR)

    else:
        print("Augmented dataset already exists:", AUGMENTED_YOLO_DATASET_DIR)

    DATA_YAML_FOR_TRAIN = AUGMENTED_DATA_YAML
    DATA_YAML = DATA_YAML_FOR_TRAIN

print("DATA_YAML_FOR_TRAIN:", DATA_YAML_FOR_TRAIN)

dataset_root_for_train = Path(yaml.safe_load(open(DATA_YAML_FOR_TRAIN, "r", encoding="utf-8"))["path"])
for split in ["train", "val", "test"]:
    verify_yolo_seg_labels(dataset_root_for_train, split)

Creating augmented dataset: /content/speedstack_3class_yolo_seg_1280_geom1_redp25

========== AUGMENTATION SUMMARY ==========
original_train_images: 1047
geometric_aug_images: 1047
red_recolor_images: 216
total_train_images: 2310
geometric_aug_objects: 3550
red_recolor_objects: 925
AUGMENTED_DATA_YAML: /content/speedstack_3class_yolo_seg_1280_geom1_redp25/data.yaml
path: /content/speedstack_3class_yolo_seg_1280_geom1_redp25
train: images/train
val: images/val
test: images/test
nc: 4
names:
- hand-eye-view-speed-stack-cup
- fallen-cup
- mouth-up-cup
- upright-cup

DATA_YAML_FOR_TRAIN: /content/speedstack_3class_yolo_seg_1280_geom1_redp25/data.yaml

[train]
image count: 2310
label file count: 2310
non-empty labels: 1980
empty/null labels: 330
missing label files: 0
max class id: 3
bad lines: 0

[val]
image count: 100
label file count: 100
non-empty labels: 86
empty/null labels: 14
missing label files: 0
max class id: 3
bad lines: 0

[test]
image count: 50
label file count: 50
non-empty l

## 8. Helper functions for training, evaluation, and saving

In [ ]:
# =========================
# Helper functions
# =========================

def run_name_for(size: str) -> str:
    return f"speedstack3class_yolo26{size}_seg_{IMG_SIZE}_epoch{EPOCHS}_{EXPERIMENT_TAG}_a100"


def weight_name_for(size: str, kind="best") -> str:
    return f"{run_name_for(size)}_{kind}.pt"


def local_run_dir_for(size: str) -> Path:
    return RUN_PROJECT / run_name_for(size)


def local_best_for(size: str) -> Path:
    return local_run_dir_for(size) / "weights" / "best.pt"


def local_last_for(size: str) -> Path:
    return local_run_dir_for(size) / "weights" / "last.pt"


def drive_best_for(size: str) -> Path:
    return DRIVE_WEIGHTS_DIR / weight_name_for(size, "best")


def drive_last_for(size: str) -> Path:
    return DRIVE_WEIGHTS_DIR / weight_name_for(size, "last")


def copy_file_if_exists(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        shutil.copy2(src, dst)
        print("copied:", src, "->", dst)
        return True
    print("[WARN] missing:", src)
    return False


def zip_dir(src_dir: Path, zip_base_path: Path):
    if not src_dir.exists():
        print("[WARN] src_dir not found:", src_dir)
        return None

    zip_base_path.parent.mkdir(parents=True, exist_ok=True)
    zip_path = shutil.make_archive(str(zip_base_path), "zip", root_dir=str(src_dir))
    print("zipped:", zip_path)
    return zip_path


def safe_float(x):
    try:
        return float(np.asarray(x).mean())
    except Exception:
        try:
            return float(x)
        except Exception:
            return np.nan


def get_speed_dict(metrics):
    speed = getattr(metrics, "speed", {}) or {}

    preprocess = safe_float(speed.get("preprocess", np.nan))
    inference = safe_float(speed.get("inference", np.nan))
    postprocess = safe_float(speed.get("postprocess", np.nan))
    total = preprocess + inference + postprocess

    fps_total = np.nan if math.isnan(total) or total <= 0 else 1000.0 / total
    fps_inference_only = np.nan if math.isnan(inference) or inference <= 0 else 1000.0 / inference

    return {
        "preprocess_ms": preprocess,
        "inference_ms": inference,
        "postprocess_ms": postprocess,
        "total_ms": total,
        "fps_total": fps_total,
        "fps_inference_only": fps_inference_only,
    }


def get_model_stats(yolo_model, imgsz=1280):
    net = yolo_model.model

    params = sum(p.numel() for p in net.parameters())
    trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)

    gflops = np.nan

    try:
        from ultralytics.utils.torch_utils import get_flops
        flops_value = get_flops(net, imgsz=imgsz)
        if flops_value is not None:
            gflops = float(flops_value)
    except Exception:
        pass

    try:
        info = net.info(verbose=False, imgsz=imgsz)
        if isinstance(info, (list, tuple)) and len(info) >= 4:
            gflops = safe_float(info[-1])
        elif isinstance(info, str):
            m = re.search(r"([0-9.]+)\s*GFLOPs", info)
            if m:
                gflops = float(m.group(1))
    except Exception:
        pass

    return {
        "params": params,
        "params_M": params / 1e6,
        "trainable_params": trainable_params,
        "trainable_params_M": trainable_params / 1e6,
        "GFLOPs": gflops,
    }


def metrics_to_row(metrics, size, split, weight_path, stats):
    speed = get_speed_dict(metrics)

    row = {
        "model_size": size,
        "model": MODEL_NAME_MAP.get(size, f"YOLOv26{size}-seg"),
        "split": split,
        "weight": str(weight_path),
        "imgsz": IMG_SIZE,
        "epochs": EPOCHS,

        "params_M": stats.get("params_M", np.nan),
        "GFLOPs": stats.get("GFLOPs", np.nan),

        "box_P": safe_float(metrics.box.p),
        "box_R": safe_float(metrics.box.r),
        "box_mAP50": safe_float(metrics.box.map50),
        "box_mAP50_95": safe_float(metrics.box.map),

        "mask_P": safe_float(metrics.seg.p),
        "mask_R": safe_float(metrics.seg.r),
        "mask_mAP50": safe_float(metrics.seg.map50),
        "mask_mAP50_95": safe_float(metrics.seg.map),
    }

    row.update(speed)
    return row


def save_dataframe(df, name: str):
    csv_path = DRIVE_TABLE_DIR / f"{name}.csv"
    xlsx_path = DRIVE_TABLE_DIR / f"{name}.xlsx"

    df.to_csv(csv_path, index=False)
    df.to_excel(xlsx_path, index=False)

    print("saved:", csv_path)
    print("saved:", xlsx_path)

    return csv_path, xlsx_path


def get_test_image_dir():
    with open(DATA_YAML, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    root = Path(data["path"])
    test_rel = data.get("test", "images/test")
    return root / test_rel

## 9. Train YOLOv26s-seg and YOLOv26m-seg

In [ ]:
# =========================
# Main loop: train -> save weights -> val/test eval -> predict all test images
# =========================

all_rows = []

test_img_dir = get_test_image_dir()
print("TEST_IMG_DIR:", test_img_dir)
print("test image count:", count_images(test_img_dir))

for size in MODEL_SIZES:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\n" + "=" * 90)
    print(f"START MODEL: {MODEL_NAME_MAP[size]}  | size={size}")
    print("=" * 90)

    model_weights = f"yolo26{size}-seg.pt"
    run_name = run_name_for(size)
    run_dir = local_run_dir_for(size)
    best_model = local_best_for(size)
    last_model = local_last_for(size)
    drive_best = drive_best_for(size)
    drive_last = drive_last_for(size)

    print("model_weights:", model_weights)
    print("run_name:", run_name)
    print("run_dir:", run_dir)
    print("drive_best:", drive_best)

    if SKIP_TRAIN_IF_DRIVE_BEST_EXISTS and drive_best.exists():
        print(f"[SKIP TRAIN] Drive best.pt already exists: {drive_best}")
        best_model_for_eval = drive_best

    else:
        model = YOLO(model_weights)

        train_results = model.train(
            data=str(DATA_YAML),
            task="segment",

            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=TRAIN_BATCH,
            patience=PATIENCE,

            optimizer="auto",
            cos_lr=True,
            warmup_epochs=5.0,
            weight_decay=0.0005,

            overlap_mask=True,
            mask_ratio=2,

            # Dataset에 red cup이 포함되어 있으므로 online color augmentation은 이전보다 약하게 설정
            hsv_h=0.03,
            hsv_s=0.25,
            hsv_v=0.15,

            # 고정 작업영역이므로 geometry augmentation도 약하게 설정
            degrees=3.0,
            translate=0.03,
            scale=0.15,
            shear=0.0,
            perspective=0.0001,
            flipud=0.0,
            fliplr=0.5,

            mosaic=0.15,
            close_mosaic=20,
            mixup=0.0,
            copy_paste=0.0,

            device=DEVICE,
            workers=8,
            amp=True,
            cache=CACHE_MODE,
            plots=True,
            save=True,
            save_period=10,

            project=str(RUN_PROJECT),
            name=run_name,
            exist_ok=True,
        )

        assert best_model.exists(), f"best.pt not found after training: {best_model}"

        copy_file_if_exists(best_model, drive_best)
        copy_file_if_exists(last_model, drive_last)

        if SAVE_RUN_ZIP:
            zip_dir(run_dir, DRIVE_RUNS_DIR / run_name)

        best_model_for_eval = drive_best if drive_best.exists() else best_model

    model = YOLO(str(best_model_for_eval))
    stats = get_model_stats(model, IMG_SIZE)
    print("model stats:", stats)

    val_name = f"{run_name}_val"
    val_metrics = model.val(
        data=str(DATA_YAML),
        task="segment",
        imgsz=IMG_SIZE,
        batch=EVAL_BATCH,
        device=DEVICE,
        split="val",
        plots=True,
        project=str(EVAL_PROJECT),
        name=val_name,
        exist_ok=True,
    )
    val_row = metrics_to_row(val_metrics, size, "val", best_model_for_eval, stats)
    all_rows.append(val_row)

    test_name = f"{run_name}_test"
    test_metrics = model.val(
        data=str(DATA_YAML),
        task="segment",
        imgsz=IMG_SIZE,
        batch=EVAL_BATCH,
        device=DEVICE,
        split="test",
        plots=True,
        project=str(EVAL_PROJECT),
        name=test_name,
        exist_ok=True,
    )
    test_row = metrics_to_row(test_metrics, size, "test", best_model_for_eval, stats)
    all_rows.append(test_row)

    pred_name = f"{run_name}_test_predict_all"
    pred_results = model.predict(
        source=str(test_img_dir),
        task="segment",
        imgsz=IMG_SIZE,
        conf=PRED_CONF,
        iou=PRED_IOU,
        device=DEVICE,
        save=True,
        save_txt=True,
        save_conf=True,
        retina_masks=True,
        project=str(PRED_PROJECT),
        name=pred_name,
        exist_ok=True,
    )

    pred_save_dir = Path(pred_results[0].save_dir)
    print("predict save dir:", pred_save_dir)

    drive_pred_model_dir = DRIVE_PRED_DIR / pred_name
    if drive_pred_model_dir.exists():
        shutil.rmtree(drive_pred_model_dir)
    shutil.copytree(pred_save_dir, drive_pred_model_dir)
    print("copied predict results to:", drive_pred_model_dir)

    df_all_intermediate = pd.DataFrame(all_rows)
    save_dataframe(df_all_intermediate, "all_metrics_intermediate")

    print(f"FINISHED MODEL: {MODEL_NAME_MAP[size]}")

    try:
        del model
    except NameError:
        pass

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll models finished.")

TEST_IMG_DIR: /content/speedstack_3class_yolo_seg_1280_geom1_redp25/images/test
test image count: 50

START MODEL: YOLOv26s-seg  | size=s
model_weights: yolo26s-seg.pt
run_name: speedstack3class_yolo26s_seg_1280_epoch250_3class_lightaug_geom1_redp25_sm_a100
run_dir: /content/runs/segment/speedstack3class_yolo26s_seg_1280_epoch250_3class_lightaug_geom1_redp25_sm_a100
drive_best: /content/drive/MyDrive/yolo26sm_3class_speedstack_result_epoch250_3class_lightaug_geom1_redp25_sm/weights/speedstack3class_yolo26s_seg_1280_epoch250_3class_lightaug_geom1_redp25_sm_a100_best.pt
Ultralytics 8.4.62 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=0.6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/speedstack_3class_yolo_s

## 10. Build validation/test result tables

In [ ]:
# =========================
# Build final validation/test tables
# =========================

df_all = pd.DataFrame(all_rows)

size_order = {"n": 0, "s": 1, "m": 2, "l": 3}
df_all["size_order"] = df_all["model_size"].map(size_order)
df_all = df_all.sort_values(["split", "size_order"]).drop(columns=["size_order"]).reset_index(drop=True)

df_val = df_all[df_all["split"] == "val"].copy().reset_index(drop=True)
df_test = df_all[df_all["split"] == "test"].copy().reset_index(drop=True)

print("========== VALIDATION RESULT TABLE ==========")
display(df_val)

print("========== TEST RESULT TABLE ==========")
display(df_test)

save_dataframe(df_all, "all_metrics_val_test")
save_dataframe(df_val, "validation_metrics")
save_dataframe(df_test, "test_metrics")

compact_cols = [
    "model", "split", "params_M", "GFLOPs",
    "box_P", "box_R", "box_mAP50", "box_mAP50_95",
    "mask_P", "mask_R", "mask_mAP50", "mask_mAP50_95",
    "inference_ms", "total_ms", "fps_total"
]

df_val_compact = df_val[compact_cols].copy()
df_test_compact = df_test[compact_cols].copy()

print("========== VALIDATION COMPACT TABLE ==========")
display(df_val_compact)

print("========== TEST COMPACT TABLE ==========")
display(df_test_compact)

save_dataframe(df_val_compact, "validation_metrics_compact")
save_dataframe(df_test_compact, "test_metrics_compact")

## 11. Plot trade-off: accuracy vs speed

In [ ]:
if len(df_test_compact) > 0:
    plt.figure(figsize=(8, 5))
    plt.scatter(df_test_compact["fps_total"], df_test_compact["mask_mAP50_95"], s=120)

    for _, r in df_test_compact.iterrows():
        plt.text(r["fps_total"], r["mask_mAP50_95"], r["model"], fontsize=11)

    plt.xlabel("FPS, total pipeline")
    plt.ylabel("Mask mAP50-95")
    plt.title("YOLOv26s/m Speed-Accuracy Trade-off, Test Set")
    plt.grid(True)
    plt.tight_layout()

    plot_path = DRIVE_TABLE_DIR / "test_speed_accuracy_tradeoff.png"
    plt.savefig(plot_path, dpi=200)
    plt.show()

    print("saved:", plot_path)

## 12. Automatic report-ready summary

In [ ]:
assert len(df_test) > 0, "df_test가 비어 있습니다."

def pct(x):
    if pd.isna(x):
        return "N/A"
    return f"{x:.3f}"

best_mask = df_test.loc[df_test["mask_mAP50_95"].idxmax()]
best_box = df_test.loc[df_test["box_mAP50_95"].idxmax()]
fastest = df_test.loc[df_test["fps_total"].idxmax()]
smallest = df_test.loc[df_test["params_M"].idxmin()]

print("========== REPORT SUMMARY ==========")
print(f"Trained models: {', '.join([MODEL_NAME_MAP[s] for s in MODEL_SIZES])}")
print(f"Classes: {', '.join(names_ref)}")
print(f"Best mask mAP50-95 model: {best_mask['model']} | mask mAP50-95={pct(best_mask['mask_mAP50_95'])}, FPS={pct(best_mask['fps_total'])}")
print(f"Best box  mAP50-95 model: {best_box['model']} | box mAP50-95={pct(best_box['box_mAP50_95'])}, FPS={pct(best_box['fps_total'])}")
print(f"Fastest model: {fastest['model']} | FPS={pct(fastest['fps_total'])}, mask mAP50-95={pct(fastest['mask_mAP50_95'])}")
print(f"Smallest model: {smallest['model']} | params={pct(smallest['params_M'])}M, mask mAP50-95={pct(smallest['mask_mAP50_95'])}")

summary_cols = [
    "model", "params_M", "GFLOPs",
    "box_mAP50_95", "mask_mAP50_95",
    "mask_R", "total_ms", "fps_total"
]

print("\n========== MODEL-WISE TEST SUMMARY ==========")
display(df_test[summary_cols].sort_values("params_M").reset_index(drop=True))

report_txt = f"""
[보고서용 요약 문단]

본 실험에서는 speed stack 컵 상태 인식을 위해 {', '.join([MODEL_NAME_MAP[s] for s in MODEL_SIZES])} 모델을 3-class instance segmentation 문제로 fine-tuning하였다. 사용한 class는 {', '.join(names_ref)}이며, COCO Segmentation 형식으로 export된 Roboflow dataset을 YOLO segmentation 형식으로 변환하여 학습에 사용하였다.

학습 데이터에는 고정된 협동로봇 작업환경을 고려하여 약한 geometric/photometric augmentation을 적용하였다. 또한 빨간 컵이 실제 dataset에 포함되어 있으므로, 기존보다 약한 red recolor augmentation을 일부 train 이미지에만 적용하였다. Background 또는 tape-only hard-negative 이미지는 빈 label을 갖는 null image로 포함되어, 테이프나 작업영역 배경에 대한 false positive를 줄이는 것을 목표로 하였다.

Validation set과 test set에 대해 bounding box 기준 Precision, Recall, mAP50, mAP50-95와 segmentation mask 기준 Precision, Recall, mAP50, mAP50-95를 측정하였다. 실제 로봇팔 perception module 적용 가능성을 확인하기 위해 이미지당 preprocess, inference, postprocess 시간을 측정하고 FPS를 계산하였다.

Test set 기준 가장 높은 Mask mAP50-95를 기록한 모델은 {best_mask['model']}이며, Mask mAP50-95={pct(best_mask['mask_mAP50_95'])}, FPS={pct(best_mask['fps_total'])}를 기록하였다. 가장 빠른 모델은 {fastest['model']}이며, FPS={pct(fastest['fps_total'])}, Mask mAP50-95={pct(fastest['mask_mAP50_95'])}를 기록하였다. 따라서 최종 모델 선택 시에는 segmentation 정확도, mask recall, 추론 속도, 모델 크기를 함께 고려해야 한다.
""".strip()

report_txt_path = DRIVE_TABLE_DIR / "report_summary.txt"
report_txt_path.write_text(report_txt, encoding="utf-8")

print("\nSaved report summary to:", report_txt_path)
print("\n" + report_txt)

## 13. 저장 위치 정리

실행이 끝나면 Google Drive에 다음 폴더가 생성됩니다.

```text
/content/drive/MyDrive/yolo26sm_3class_speedstack_result_epoch250_3class_lightaug_geom1_redp25_sm
```

주요 저장물은 다음과 같습니다.

```text
weights/
  speedstack3class_yolo26s_seg_1280_epoch250_..._best.pt
  speedstack3class_yolo26s_seg_1280_epoch250_..._last.pt
  speedstack3class_yolo26m_seg_1280_epoch250_..._best.pt
  speedstack3class_yolo26m_seg_1280_epoch250_..._last.pt

tables/
  validation_metrics.csv
  test_metrics.csv
  validation_metrics_compact.csv
  test_metrics_compact.csv
  all_metrics_val_test.csv
  report_summary.txt
  test_speed_accuracy_tradeoff.png

runs_zip/
  각 모델별 전체 training run folder zip

predict_images/
  각 모델별 test 전체 이미지에 대한 prediction 결과
```

## 14. 실험 해석 포인트

이 노트북의 결과는 다음을 확인하기 위한 것입니다.

- `fallen-cup`, `upright-cup`, `mouth-up-cup` 세 상태를 YOLOv26s/m-seg가 안정적으로 구분하는가?
- hard-negative 이미지를 추가한 뒤, 테이프나 배경 물체에 대한 false positive가 줄어드는가?
- small과 medium 중 어떤 모델이 Mask mAP50-95, Mask Recall, FPS 관점에서 더 적절한가?
- 고정된 협동로봇 작업영역에서는 medium의 정확도가 필요한가, 아니면 small의 속도와 성능이 충분한가?

로봇팔 컵쌓기 task에서는 bounding box보다 **mask 품질과 class별 recall**이 중요합니다.